# Explaining RL GRPO 

The goal is to walk through the Group Relative Policy Optimization (GRPO) algorithm approach to reinforcement learning (RL). GRPO was famously introduced by [DeepSeek](https://arxiv.org/pdf/2402.03300) and has been one of the backbones for reasoning models. GRPO is a variant of Proximal Policy Optimization (PPO) but, contrary to PPO, it completely eliminates the need for a separate critic (value) model. GRPO does this by generating multiple responses (group) and uses those responses to calculate advantage without a separate, heavy value model.

We'll focus on applying GRPO not to an LLM, but instead to a simple number sorting model. By still having a model that generates the next token using logits, we can see how GRPO works while having a tractable policy, aka model. 

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import copy

## Data Prep

For our model, we will want to take in a list of integers and the model's goal is to learn how to sort them in descending order. E.g. input of `[0,2,1]` should output as `[2,1,0]`.  We'll start by randomly generating a set of inputs. You'll notice that we're okay with repeatitive values as in a real examples, tokens can repeat.  

In [2]:
torch.manual_seed(14)
vocab_size = 6
embedding_dim = 8
prompt_length = response_length = 5
batch = 2

In [3]:
prompt_test = torch.randint(0, vocab_size, (batch, prompt_length))
prompt_test.shape, prompt_test

(torch.Size([2, 5]),
 tensor([[3, 0, 4, 2, 3],
         [1, 2, 2, 2, 4]]))

## Policy (aka Model)

Now we'll create a simple model for our task.  If we were using an LLM this would be the actual LLM. In the RL world, this is the "Policy" that we're learning. For our task, we'll build a simple non-autoregressive encoder-decoder. The encoder embeds each prompt token, adds positional embeddings, passes them through a linear layer, and then sums across the sequence to compress the entire prompt into a single fixed-size vector. The decoder takes that single vector, broadcasts it across every response position, adds response-position embeddings so each slot knows where it sits, and projects to vocab logits. 

With our model, response length output tokens are predicted in one parallel forward pass rather than one-at-a-time.   

In [4]:
class Model(nn.Module):
    def __init__(self, vocab_size, embd_dim, prompt_len, response_len):
        super().__init__()
        self.response_len = response_len
        
        self.embedding = nn.Embedding(vocab_size, embd_dim)
        self.prompt_pos_embedding = nn.Embedding(prompt_len, embd_dim)
        self.response_pos_embedding = nn.Embedding(response_len, embd_dim)
        self.encoder = nn.Linear(embd_dim, embd_dim)
        self.decoder = nn.Linear(embd_dim, embd_dim)
        self.output = nn.Linear(embd_dim, vocab_size)
        
    def forward(self, prompts):
        batch_size, prompt_len = prompts.shape
        
        # generate positions representations for input and output
        prompt_positions = torch.arange(prompt_len)
        response_positions = torch.arange(self.response_len)

        embeddings = self.embedding(prompts)
        embeddings =  embeddings + self.prompt_pos_embedding(prompt_positions)            
        encoded = self.encoder(embeddings).sum(dim=1)
        
        resp_position_embed = self.response_pos_embedding(response_positions).unsqueeze(0)
        decoded = self.decoder(encoded).unsqueeze(1)  
        decoded = decoded + resp_position_embed
        logits = self.output(decoded)
        return logits


## Step by Step First Pass
Our first walk through will go through step by step what a single pass through an RL loop would be.  This is the same as the "Algorithm 1: Iterative Group Relative Policy Optimization" in the GRPO paper.  We wil show: 
1. caching of the policy so that we have a reference to act as "reference log probabilities" to compute the KL (Kullback-Leibler) penalty
2. sampling a set of responses for our prompts
3. calcualting the rewards and advantage. The advantage is the relative signal  calculated as the reward minus some baseline with the goal of indicating more clearly if the generated response is better or worse than expected. Positive advantages pushe probability up, and drive the gradient to reinforce those pathways while negative values pusht he model to weaken those pathways. 
4. based on the responses calculating reference log probabilities based on the reference policy
5. based on the responses calculating an initial "old" log probability based on the policy.  Note that we update the reference policy only every N epochs so the ref logprobs and the old log probs will diverge between the those epochs. 
6. For every N steps in an "epoch" we
    1. calculate the current step logprobs using the current policy
    2. calculate the loss using the logprobs, deltas.
    3. add the kl penalty to the loss using the comparison of the logprobs and the reference log probs
    4. update the policy (driven by the delta, logprobs, and kl penalty)


We will add our own wrinkle on this as we go through it showing some different ways to calculate rewards, advantage, and loss.  While ultimately we'll pick the ones closest to GRPO for the final example, we want to add a few of these in to play with different ideas we've seen. 

In [5]:
records = []
ref_log_probs = None
ref_model = None
old_log_probs = None
lossi = []
rewardi = []

### Policy Initiailzation
We'll start by initializing the policy and our optimizer to help with stepping through training.  The Adam optimizer allows us to update the policy's trainable parameters updates them using Adam's adaptive per-parameter learning rates (scaled by lr) based on the gradients populated by loss.backward(). This is a bit different than a typical training run where you'd also pair a LR scheduler.   

In [6]:
lr = 1e-3
policy_model = Model(vocab_size=vocab_size, embd_dim=10, prompt_len=prompt_length, response_len=response_length)
optimizer = torch.optim.Adam(policy_model.parameters(), lr=lr)

### KL: Ref policy copy

Our next step is to make a copy of the policy that isn't updated as frequently for us to calculate the KL penalty. The KL penalty's job is to stop the policy from drifting too far from a trusted distribution.  This distribution is often the pretrained or supervised model you started with. If we compared the current policy against itself, KL would always be zero and the penalty would do nothing. If we compared only against the previous step's model, we'd drift but slowly since the ref policy would drift along with the main policy.
In practice, by freezing a copy (and only refreshing it occasionally, or never, depending on the setup), we get a stable reference point: the KL term measures how much the live policy has diverged from that anchor and pulls it back, which keeps reward-hacking and mode collapse in check while still letting the policy improve between refreshes. 

In our example we'll use the the raw initialized policy. Here though we'll be a bit different from an LLM style usage as we'll allow the ref policy to drift since we've initialized a random set of noise.  

In [7]:
ref_policy = copy.deepcopy(policy_model)

### Generate Responses

Now we're ready to generate responses.  As a reminder, our test has a dimension of `[4, 5]` and we have a vocab size of 6.  Our model outputs logits, which is a pseudo-probability of each token for each position. After we generate them we'll also remove the batch dimension to make them easier to work with. A final step is to turn the output into probabilities.  We do this by taking the softmax using: 
$$
p_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}
$$
where $z$ is the logit vector along the last dim and $V$ is its size (e.g. vocab_size). This will now allow us to do sampling from the probabilities to generate an output from the distribution. 

In [8]:
logits = policy_model(prompt_test)
logits.shape, logits

(torch.Size([2, 5, 6]),
 tensor([[[-1.1137,  0.2320, -0.9153, -1.2998,  0.8457, -1.2956],
          [-0.5200, -0.6550,  0.7312,  0.2488, -0.5351, -2.1397],
          [-0.7925, -0.3202, -0.2491, -0.2959,  0.8456, -0.7270],
          [-0.8079, -0.3992, -0.0129,  0.1981,  0.0600, -0.7187],
          [-0.2375, -0.5434, -0.6405, -0.2932,  0.4465, -0.1970]],
 
         [[-0.8538,  0.0724, -0.9676, -1.3099,  1.0302, -1.1925],
          [-0.2600, -0.8146,  0.6790,  0.2387, -0.3505, -2.0366],
          [-0.5325, -0.4798, -0.3014, -0.3060,  1.0302, -0.6239],
          [-0.5480, -0.5588, -0.0652,  0.1880,  0.2445, -0.6156],
          [ 0.0225, -0.7030, -0.6927, -0.3033,  0.6310, -0.0940]]],
        grad_fn=<ViewBackward0>))

In [9]:
logits = logits.reshape(-1, logits.size(-1))
logits.shape, logits

(torch.Size([10, 6]),
 tensor([[-1.1137,  0.2320, -0.9153, -1.2998,  0.8457, -1.2956],
         [-0.5200, -0.6550,  0.7312,  0.2488, -0.5351, -2.1397],
         [-0.7925, -0.3202, -0.2491, -0.2959,  0.8456, -0.7270],
         [-0.8079, -0.3992, -0.0129,  0.1981,  0.0600, -0.7187],
         [-0.2375, -0.5434, -0.6405, -0.2932,  0.4465, -0.1970],
         [-0.8538,  0.0724, -0.9676, -1.3099,  1.0302, -1.1925],
         [-0.2600, -0.8146,  0.6790,  0.2387, -0.3505, -2.0366],
         [-0.5325, -0.4798, -0.3014, -0.3060,  1.0302, -0.6239],
         [-0.5480, -0.5588, -0.0652,  0.1880,  0.2445, -0.6156],
         [ 0.0225, -0.7030, -0.6927, -0.3033,  0.6310, -0.0940]],
        grad_fn=<ViewBackward0>))

In [10]:
probs = torch.softmax(logits, dim=-1)
probs.shape, probs

(torch.Size([10, 6]),
 tensor([[0.0675, 0.2592, 0.0823, 0.0560, 0.4788, 0.0563],
         [0.1148, 0.1003, 0.4013, 0.2477, 0.1131, 0.0227],
         [0.0821, 0.1316, 0.1413, 0.1349, 0.4224, 0.0876],
         [0.0915, 0.1377, 0.2026, 0.2502, 0.2179, 0.1000],
         [0.1569, 0.1155, 0.1049, 0.1484, 0.3109, 0.1634],
         [0.0810, 0.2045, 0.0723, 0.0513, 0.5331, 0.0577],
         [0.1458, 0.0837, 0.3727, 0.2400, 0.1331, 0.0247],
         [0.0975, 0.1028, 0.1229, 0.1223, 0.4654, 0.0890],
         [0.1131, 0.1119, 0.1833, 0.2361, 0.2499, 0.1057],
         [0.1844, 0.0893, 0.0902, 0.1331, 0.3389, 0.1641]],
        grad_fn=<SoftmaxBackward0>))

### Sample from distribution
Now that we have probabilities, we need to sample from the probabilities to generate a set of responses possible based on the current policy. These responses will be the "group" in GRPO.   For the sake of this calculation, we'll sample 3 and then re-insert in the batch so we can easily compare against the prompts. 

In [11]:
num_responses = 3

In [12]:
responses = torch.multinomial(probs, num_responses, replacement=True)
responses.shape, responses

(torch.Size([10, 3]),
 tensor([[5, 4, 4],
         [2, 2, 2],
         [4, 3, 4],
         [4, 2, 4],
         [4, 5, 1],
         [1, 4, 4],
         [0, 1, 2],
         [0, 4, 4],
         [3, 3, 2],
         [3, 0, 0]]))

In [13]:
responses = responses.reshape(batch, -1, num_responses).permute(0, 2, 1)
responses.shape, responses

(torch.Size([2, 3, 5]),
 tensor([[[5, 2, 4, 4, 4],
          [4, 2, 3, 2, 5],
          [4, 2, 4, 4, 1]],
 
         [[1, 0, 0, 3, 3],
          [4, 1, 4, 3, 0],
          [4, 2, 4, 2, 0]]]))

### Calculate Reward
Now we're ready to calculate the reward.  The reward scores each sampled response in absolute terms based on the rules we define, giving you a scalar quality signal per response. With GRPO we end up converting these rewards into advantages where we look at the reward and normalize against each other.  This means the reward doesn't drive the gradient directly but rather the reward drives how samples sit relative to the rest of the responses in the group so we know which ones to upweight and downweight. 

With RL, getting the reward right is a bit of an art form.  If you make the reward too sparse, the model may never learn. If you make it too free, the model can fall into a local min.  This is why it's good to make it easy to compare different reward models as part of your tuning.  To make this easy, we'll create a scaffold to apply different reward functions that way we can see how the different frameworks impact the scoring.  

In [14]:
def compute_reward(prompts, responses, reward_fn, debug=False):
    batch, n_resp, _ = responses.shape
    rewards = torch.empty(batch, n_resp, dtype=torch.float32)
    for i in range(batch):
        for j in range(n_resp):
            rewards[i,j] = reward_fn(prompts[i,:], responses[i,j,:])
            if debug:
                print(f'prompt {prompts[i,:]} | response {responses[i,j,:]} | reward {rewards[i,j]}')
    return rewards

#### Distance reward

Our first reward is a super simple reward function. We see how many positions match the true sorted order of our prompt and give a point for each one.  We calculate it as
$$
R(p, r) = \sum_{i=1}^{n} 1[r_i = g_i], \qquad g = \text{sort}_{\downarrow}(p) 
$$
where $p$ is the prompt, $r$ the response (both length $n$), $g$ is $p$ sorted in descending order, and $\mathbb{1}[\cdot]$ is 1 when the condition holds and 0 otherwise. So the reward issimply the count of positions where the response matches the sorted ground truth.  Even with our current untrained model you'll see that for each example prompt we actually get up to a score of 3 of a possible max of 5.  Also you can see that there's no penalties for the model outputting the wrong value or the wrong duplicates. 

In [15]:
def sort_distance_reward(prompt, response):
    assert len(prompt) == len(response)
    ground_truth = sorted(prompt, reverse=True)
    match = sum(1 for x,y in zip(response, ground_truth) if x == y)
    return match

In [16]:
compute_reward(prompt_test, responses, sort_distance_reward, debug=True)

prompt tensor([3, 0, 4, 2, 3]) | response tensor([5, 2, 4, 4, 4]) | reward 0.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 3, 2, 5]) | reward 3.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 4, 4, 1]) | reward 1.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([1, 0, 0, 3, 3]) | reward 0.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 1, 4, 3, 0]) | reward 1.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 2, 4, 2, 0]) | reward 3.0


tensor([[0., 3., 1.],
        [0., 1., 3.]])

#### Sort Inclusion reward
Our next reward focuses on rewarding both that the answer includes the values from the input, and that it's started grasping the concept of sorting. We calcluate sort inclusion ordering reward as:
$$
R(p, r) = \underbrace{\sum_{i=1}^{n} 1[p_i \in r]}_{\text{inclusion}} + \underbrace{\sum_{i=1}^{n-1} 1[r_i > r_{i+1}]}_{\text{ordering}}
$$
The first term counts how many prompt tokens appear anywhere in the response (membership, order-agnostic), and the second counts adjacent pairs in the response that are strictly decreasing. Note that we're still not adding any penalty, just rewarding that the model learns the benefit of including the right values and learns the concept of ordering, not specifically the right places. With this framework, a perfect answer scores a 9 (5 for inclusion, 4 for sorting).  We can see that our model actually got up to a 7 and, with this calcuation, the two responses that both scored a 3 on the distance reward now have separation.  You can also see that every response got some reward in this case so we densified the reward.

In [17]:
def sort_inclusion_ordering_reward(prompt, response):
    assert len(prompt) == len(response)
    inclusion_reward = sum(1 for x in prompt if x in response)
    order_reward = sum(1 for x,y in zip(response[:-1],response[1:]) if x > y)
    return inclusion_reward + order_reward

In [18]:
compute_reward(prompt_test, responses, sort_inclusion_ordering_reward, debug=True)

prompt tensor([3, 0, 4, 2, 3]) | response tensor([5, 2, 4, 4, 4]) | reward 3.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 3, 2, 5]) | reward 6.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 4, 4, 1]) | reward 4.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([1, 0, 0, 3, 3]) | reward 2.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 1, 4, 3, 0]) | reward 5.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 2, 4, 2, 0]) | reward 7.0


tensor([[3., 6., 4.],
        [2., 5., 7.]])

#### Sort Inclusion Penalty Reward
Our final reward prompt we'll explore will add on a penalty. We want to really deincentivize the model from including the wrong values so we'll start deducting points for that. We also know that our input can have the same value repeated and don't want the model to fret over it so we'll allow the model to get a point for putting the same value together. Our new reward is calucated as:
$$
R(p, r) = \underbrace{\sum_{i=1}^{n} 1[p_i \in r]}_{\text{inclusion}} - \underbrace{\sum_{i=1}^{n} 1[r_i \notin p]}_{\text{exclusion penalty}} + \underbrace{\sum_{i=1}^{n-1} 1[r_i \geq r_{i+1}]}_{\text{ordering}}
$$
Our first term is the same as in the inclusion reward. Our exclusion penatly is new and adds in a point for ever value that's in the response and not in the prompt. The ordering reward is updated to include a point for the values being the same next to each other. 

Now this might see like by far the best, after all, it's the most complicated.  But rewards like this can have many different edge cases that, as you train a model emerge. For instance, this reward model will dilute the push for the policy to learn purely ordering.   With this complex reward you can see the first response actually gets even more points due to our favoring of adjacencies. The last example shows a hack the model can learn: if it just generates one value for any repeat in the prompt, it gets credit for including every repeat. 

In [19]:
def sort_inclusion_penalty_reward(prompt, response):
    assert len(prompt) == len(response)
    inclusion_reward = sum(1 for x in prompt if x in response)
    exclusion_penalty = sum(-1 for x in response if x not in prompt)
    order_reward = sum(1 for x,y in zip(response[:-1],response[1:]) if x >= y)
    return inclusion_reward + exclusion_penalty + order_reward

In [20]:
compute_reward(prompt_test, responses, sort_inclusion_penalty_reward, debug=True)

prompt tensor([3, 0, 4, 2, 3]) | response tensor([5, 2, 4, 4, 4]) | reward 4.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 3, 2, 5]) | reward 5.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 4, 4, 1]) | reward 4.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([1, 0, 0, 3, 3]) | reward 0.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 1, 4, 3, 0]) | reward 3.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 2, 4, 2, 0]) | reward 6.0


tensor([[4., 5., 4.],
        [0., 3., 6.]])

#### Fixed complex Reward
We'll go ahead an now fix the reward hack we had in the previous reward and calculate it as:  
$$
R(p, r) = \underbrace{\sum_v \min(c_v(p), c_v(r))}_{\text{inclusion}} - \underbrace{\sum_v \max(0, c_v(r) - c_v(p))}_{\text{exclusion penalty}} + \underbrace{\sum_{i=1}^{n-1} 1[r_i \geq r_{i+1}]}_{\text{ordering}}
$$

This now tweaks the inclusion and exclusion reward to no longer allow duplicate values to get benefits.  You'll notice that the exclusion is the inverse of the inclusion reward. This may have some unintended consequences, but we'd have to think harder to know what they are. WE can see that this acutally has a similar distribuion to the previous reward, just depressed due to the inaccurate values. 

In [21]:
def count_overlap(l1, l2):
    l2 = list(l2) # added to not mutate the original
    overlap = 0
    for x in l1:
        if x in l2:
            l2.remove(x)
            overlap += 1
    return overlap

In [22]:
def sort_complex_reward(prompt, response):
    assert len(prompt) == len(response)
    inclusion_reward = count_overlap(prompt, response)
    exclusion_penalty = inclusion_reward - len(response)
    order_reward = sum(1 for x,y in zip(response[:-1],response[1:]) if x >= y)
    return inclusion_reward + exclusion_penalty + order_reward

In [23]:
compute_reward(prompt_test, responses, sort_complex_reward, debug=True)

prompt tensor([3, 0, 4, 2, 3]) | response tensor([5, 2, 4, 4, 4]) | reward 2.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 3, 2, 5]) | reward 3.0
prompt tensor([3, 0, 4, 2, 3]) | response tensor([4, 2, 4, 4, 1]) | reward 2.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([1, 0, 0, 3, 3]) | reward 0.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 1, 4, 3, 0]) | reward 2.0
prompt tensor([1, 2, 2, 2, 4]) | response tensor([4, 2, 4, 2, 0]) | reward 4.0


tensor([[2., 3., 2.],
        [0., 2., 4.]])

**We'll acutally use our complex reward for this forward pass**

In [24]:
rewards = compute_reward(prompt_test, responses, sort_complex_reward)
rewards.shape, rewards

(torch.Size([2, 3]),
 tensor([[2., 3., 2.],
         [0., 2., 4.]]))

### Calculate Advantage $\delta$ for reward R
Tweaking the reward calculation can do a lot, but ultimately it tells the policy just if a response is good in absolute terms. In all our reward functions the output is positive meaning that the model only gets reinforced if we used the raw reward since the positive values will update gradients.  A better approach would be to both update the good answers with positive gradients and de-emphasize the bad answers with negative gradients. To do this, we calcualte the **Advantage**. The advantage function in RL measures how much better a specific action is compared to the average action available in a given state, represented as: 
$$
A(s, a) = Q(s, a) - V(s)
$$
where the advantage $A$ is the difference between the action-value $Q$ and the state-value $V$.  In GRPO the way we look at advantage is by comparing the reward of a specific example in relation to the reward other examples in the batch were able to achieve. Similar to the reward function, tuning the advantage function can be a bit of an art form and should be treated as an exploration. We'll explore a few different types of advantage functions that can be used so you can see how the different functions impact the advantage. We will use the `sort_complex_reward` reward function as the base of the advantage to simplify comparison. 

*Note that in many GRPO examples you'll see this called the delta.  Advantage is a term borrowed from traditional RL and I'll reuse it to emphasize what it's measuring*

In [25]:
def compute_advantage(rewards, delta_fn, debug=False):
    delta = delta_fn(rewards)
    if debug:
        for i in range(len(rewards)):
            print(f'reward {rewards[i]} | delta {delta[i]}')
    return delta

#### Reward aka No Adavantage

The first example we'll show is a simple pass-through where we just use the reward direclty.   

In [ ]:
def reward_advantage(rewards):
    return rewards

In [ ]:
compute_advantage(rewards, reward_advantage, debug=True)

### Centered deltas

In [ ]:
def centered_delta(rewards):
    mean_rewards = rewards.mean(dim=-1, keepdim=True)
    centered_rewards = rewards - mean_rewards
    return centered_rewards

In [ ]:
compute_delta(rewards, centered_delta, debug=True)

### Noramlized deltas

In [ ]:
def normalized_delta(rewards):
    mean_rewards = rewards.mean(dim=-1, keepdim=True)
    std_rewards = rewards.std(dim=-1, keepdim=True)
    centered_rewards = rewards - mean_rewards
    normalized_rewards = centered_rewards / (std_rewards + 1e-5)
    return normalized_rewards

In [ ]:
compute_delta(rewards, normalized_delta, debug=True)

### Custom Max/Min deltas
only keeps max values and negative mins. We don't wan positive mins since those will trigger positive gradients

In [ ]:
def max_min_delta(rewards):
    max_reward = rewards.max(dim=-1, keepdim=True)[0]
    min_reward = rewards.min(dim=-1, keepdim=True)[0]

    is_max = rewards == max_reward
    is_min_neg = (rewards == min_reward) & (min_reward < 0)
    mask = is_max | is_min_neg
    max_min_reward = torch.where(mask, rewards, torch.zeros_like(rewards))
    
    return max_min_reward

In [ ]:
compute_delta(rewards, max_min_delta, debug=True)

### We'll use normalized for now

In [ ]:
deltas = compute_delta(rewards, normalized_delta)
deltas.shape, deltas

### Ref Log Prob
only when needed for KL penalty

In [ ]:
with torch.no_grad():
    ref_logits = ref_model(prompt_test)

ref_logits.shape, ref_logits

In [ ]:
with torch.no_grad():
    ref_log_probs = F.log_softmax(ref_logits, dim=-1).unsqueeze(1)
ref_log_probs.shape, ref_log_probs

In [ ]:
with torch.no_grad():
    ref_log_probs = ref_log_probs.expand(-1, responses.size(1), -1, -1)
ref_log_probs.shape, ref_log_probs

In [ ]:
with torch.no_grad():
    ref_log_probs = ref_log_probs.gather(dim=-1, index=responses.unsqueeze(-1)).squeeze(-1)
ref_log_probs.shape, ref_log_probs

**Adding a touch of random noise to show diff**

In [ ]:
with torch.no_grad():
    noise_scale = 1e-2
    ref_log_probs = ref_log_probs + noise_scale * torch.randn_like(ref_log_probs)
ref_log_probs

### Old Log Probs

no noise this time to show it's the same for first loop but not second. 

In [ ]:
def compute_log_probs(prompts, responses, model):
    logits = model(prompts)
    log_probs = F.log_softmax(logits, dim=-1).unsqueeze(1)
    log_probs = log_probs.expand(-1, responses.size(1), -1, -1)
    log_probs = log_probs.gather(dim=-1, index=responses.unsqueeze(-1)).squeeze(-1)
    return log_probs

In [ ]:
with torch.no_grad():
    old_log_probs = compute_log_probs(prompt_test, responses, model)
old_log_probs.shape, old_log_probs
    

## One Step Through

### Sample log probs and compare with responses

In [ ]:
log_probs = compute_log_probs(prompt_test, responses, model)
log_probs.shape, log_probs

### Loss

In [ ]:
def compute_loss(log_probs, deltas, loss_fn, old_log_probs=None, debug=False):
    raw_loss = loss_fn(log_probs, deltas, old_log_probs)
    loss = raw_loss.mean()
    if debug:
        print(f'loss {loss}\n{raw_loss}')
    return loss

#### Naive Loss

In [ ]:
def naive_loss(log_probs, deltas, old_log_probs):
    return -(log_probs * deltas.unsqueeze(-1))

In [ ]:
compute_loss(log_probs, deltas, naive_loss, old_log_probs=old_log_probs, debug=True)

#### Unclipped Loss

In [ ]:
def unclipped_loss(log_probs, deltas, old_log_probs):
    ratios = torch.exp(log_probs - old_log_probs)
    return -(ratios * deltas.unsqueeze(-1))

In [ ]:
compute_loss(log_probs, deltas, unclipped_loss, old_log_probs=old_log_probs, debug=True)

#### Clipped Loss

In [ ]:
def clipped_loss(log_probs, deltas, old_log_probs):
    eps = 0.1
    uc_ratios = torch.exp(log_probs - old_log_probs)
    uc_loss = uc_ratios * deltas.unsqueeze(-1)
    c_ratios = torch.clamp(uc_ratios, min=1-eps, max=1+eps)
    c_loss = c_ratios * deltas.unsqueeze(-1)
    return -torch.minimum(uc_loss, c_loss)

In [ ]:
compute_loss(log_probs, deltas, clipped_loss, old_log_probs=old_log_probs, debug=True)

#### we'll use clipped loss for now

In [ ]:
loss = compute_loss(log_probs, deltas, clipped_loss, old_log_probs=old_log_probs)
loss

### KL Penalty

In [ ]:
kl_penalty = 0.01

In [ ]:
def compute_kl_penalty(log_probs, ref_log_probs):
    q_p = torch.exp(ref_log_probs-log_probs)
    log_q_p = ref_log_probs-log_probs
    return (q_p - log_q_p - 1).sum(dim=-1).mean()

In [ ]:
klp_raw = compute_kl_penalty(log_probs, ref_log_probs)
klp_raw

In [ ]:
loss += kl_penalty * klp_raw
loss

### Update Model Weights

In [ ]:
optimizer.zero_grad()

In [ ]:
loss.backward()

In [ ]:
optimizer.step()

## Step 2 (show update impact)

### LogProbs

In [ ]:
log_probs = compute_log_probs(prompt_test, responses, model)
log_probs.shape, log_probs

### Loss

In [ ]:
loss = compute_loss(log_probs, deltas, clipped_loss, old_log_probs=old_log_probs)
loss

### KL Penalty

In [ ]:
klp_raw = compute_kl_penalty(log_probs, ref_log_probs)
klp_raw

In [ ]:
loss += kl_penalty * klp_raw
loss

## Put it all together in a loop

In [ ]:
def generate_responses(prompts, model, num_responses=1):
    logits = model(prompts)
    batch_size = prompts.shape[0]

    logits = logits.reshape(-1, logits.size(-1))
    probs = torch.softmax(logits, dim=-1)
    
    # sample from logits to get responses
    responses = torch.multinomial(probs, num_responses, replacement=True)
    responses = responses.reshape(batch_size, -1, num_responses).permute(0, 2, 1)

    return responses

In [ ]:
def run_policy_gradient(prompts, vocab_size, epochs=200, steps_per_epoch=10, ref_model_period=10, 
                        num_responses=10, delta_fn=reward_delta,loss_fn=naive_loss,kl_penalty=0.0,
                        reward_fn=sort_complex_reward, lr=1e-3, debug=False):
    torch.manual_seed(14)
    
    prompt_length = response_length = prompts.shape[1]
    model = Model(vocab_size=vocab_size, embd_dim=8, prompt_len=prompt_length, response_len=response_length)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    records = []
    ref_log_probs = None
    ref_model = None
    old_log_probs = None
    lossi = []
    rewardi = []
    
    for epoch in tqdm(range(epochs), desc='epoch'):
        # If using KL penalty, need to get the reference model (freeze it every few epochs)
        if kl_penalty != 0:
            if epoch % ref_model_period == 0:
                ref_model = copy.deepcopy(model)
        # Sample responses and evaluate their rewards
        responses = generate_responses(prompts=prompts, model=model, num_responses=num_responses) 
        rewards = compute_reward(prompts, responses, reward_fn)
        deltas = compute_delta(rewards, delta_fn)
        if kl_penalty != 0:  # Compute under the reference model
            with torch.no_grad():
                ref_log_probs = compute_log_probs(prompts=prompts, responses=responses, model=ref_model)  # [batch trial]
        if loss_fn != naive_loss:  # Compute under the current model (but freeze while we do the inner steps)
            with torch.no_grad():
                old_log_probs = compute_log_probs(prompts=prompts, responses=responses, model=model)  # [batch trial]
        # Take a number of steps given the responses
        for step in range(steps_per_epoch):
            log_probs = compute_log_probs(prompts=prompts, responses=responses, model=model)  # [batch trial]
            loss = compute_loss(log_probs, deltas, loss_fn, old_log_probs=old_log_probs)
            if kl_penalty != 0:
                loss += kl_penalty * compute_kl_penalty(log_probs=log_probs, ref_log_probs=ref_log_probs)
            # Print information
            if (step == 0 or step == steps_per_epoch) and debug:
                print(f'e {epoch}, s {step}, loss {loss}')
            global_step = epoch * steps_per_epoch + step
            records.append({'epoch': epoch, 'step': global_step, 'loss': loss.item(), 'mean_reward': rewards.mean().item()})
            lossi.append(loss.item())
            rewardi.append(rewards.mean().item())
            # Backprop and update parameters
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return lossi, rewardi

In [ ]:
def plot_run_and_reward(run, reward):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(run)
    #axes[0].set_yscale('log')
    axes[0].set_title('Encoder Training Loss')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')

    axes[1].plot(reward)
    axes[1].set_title('Reward')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')

    plt.tight_layout()
    plt.show()

### GRPO: Clipped Run, Normalized Rewards, KL

GRPO's defining moves are
1. group-relative advantages computed as mean/std-normalized rewards across the G samples per prompt. `normalized` deltas
2. the PPO-style clipped ratio surrogate. `clipped` loss
3. a KL penalty to a periodically-refreshed reference policy. `kl_penalty` over zero

In [ ]:
batch = 10
grpo_prompt = torch.randint(0, vocab_size, (batch, prompt_length))
grpo_prompt

In [ ]:
loss, reward = run_policy_gradient(prompts=grpo_prompt, vocab_size = vocab_size, 
                                   delta_fn=normalized_delta, loss_fn=clipped_loss, 
                                   kl_penalty=0.01, epochs=800)

plot_run_and_reward(loss, reward)

### Clipped Run, Max_Min Rewards, KL

In [ ]:
mm_loss, mm_reward = run_policy_gradient(prompts=grpo_prompt, vocab_size = vocab_size, 
                                   delta_fn=max_min_delta, loss_fn=clipped_loss, 
                                   kl_penalty=0.01, epochs=800)

plot_run_and_reward(mm_loss, mm_reward)